# Experiment 7 — Semantic Search + Extractive QA
**Roll No.: 09 | Topic: Philosophy**

True sentence embeddings + cosine similarity + extractive QA.

In [ ]:
from pathlib import Path
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline


In [ ]:
DATA_DIR = Path('data')
documents = []
for path in sorted(DATA_DIR.glob('*.txt')):
    documents.append({'title': path.stem.replace('_', ' ').title(), 'content': path.read_text(encoding='utf-8')})
print('Documents:', len(documents))

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
document_embeddings = embedding_model.encode([d['content'] for d in documents], convert_to_numpy=True, normalize_embeddings=True)
print('Embedding shape:', document_embeddings.shape)

In [ ]:
question = 'What does Stoicism teach about controlling emotions?'
query_embedding = embedding_model.encode([question], convert_to_numpy=True, normalize_embeddings=True)
scores = cosine_similarity(query_embedding, document_embeddings)[0]
ranking = np.argsort(scores)[::-1]
for i in ranking:
    print(documents[i]['title'], round(float(scores[i]), 4))

In [ ]:
best_index = ranking[0]
context = documents[best_index]['content']
qa = pipeline('question-answering', model='distilbert-base-cased-distilled-squad', tokenizer='distilbert-base-cased-distilled-squad')
answer = qa(question=question, context=context)
print('Selected document:', documents[best_index]['title'])
print('Answer:', answer['answer'])
print('QA score:', answer['score'])